# DeepEpilepsy: дообучение на всём NMT, аугментации, 3 запуска → турецкий набор

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-20 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Усилить перенос NMT → Турция: всё NMT train вместо подвыборки, до 6 эпох, **аугментации реальных записей**, **3 запуска** с разными начальными значениями (среднее ± разброс), ансамбль запусков и ансамбль с простыми признаками |
| **Данные** | Обучение: NMT train (Kaggle `mhmdrdwn/nmt-eeg`, до 8 окон по 30 с на запись); выбор эпохи — 15 % записей NMT train; проверка: NMT eval и **турецкий набор** (`buraktaci/turkish-epilepsy`, 50 + 71, не виден при обучении) |
| **Аугментации** | После нормировки окна: гауссов шум (σ = 0,1), зануление 1–2 случайных каналов (p = 0,5), зеркальная замена полушарий (Fp1↔Fp2, F3↔F4 … , p = 0,5), циклический сдвиг во времени до 2 с. Масштаб амплитуды не используем — его стирает нормировка |
| **Среда** | Kaggle, пакетный запуск, GPU T4 |

Предыдущий запуск (подвыборка 1 020 записей, 3 эпохи, без аугментаций, 1 запуск): Турция 0,66, NMT eval 0,80.

## 1. Модель и общие функции

In [ ]:
import os, glob, time, subprocess, importlib.util
import numpy as np, pandas as pd, torch, torch.nn.functional as F, mne, scipy.io, scipy.signal as ss
from huggingface_hub import hf_hub_download
from sklearn.metrics import roc_auc_score
mne.set_log_level("ERROR")

REPO = "/kaggle/working/dl_epilepsy_reeg"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://gitlab.com/chum-epilepsy/dl_epilepsy_reeg.git", REPO], check=True)
spec = importlib.util.spec_from_file_location("de_transformers", f"{REPO}/src/epileptology/ml/models/transformers.py")
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)

SFREQ, SEG = 200, 30
L = SFREQ * SEG
W_PATH = hf_hub_download("emilelemoine/DeepEpilepsy_v1", "DeepEpilepsy_large_50_30s_augment.pt")
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def new_model():
    m = de.ViT1d(n_class=2, in_channels=19, n_timepoints=L, patch_size=50, tokenizer="conv1d",
                 n_layers=6, n_heads=8, dim_hidden=512, dim_mlp=512, dropout=0.25, attention_dropout=0.25)
    m.load_state_dict(torch.load(W_PATH, map_location="cpu", weights_only=True), strict=True)
    return m.to(DEV)

def scale(x):
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-20)

def predict(model, Ws):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        for W in Ws:
            p = []
            for b in range(0, len(W), 64):
                x = torch.tensor(scale(W[b:b + 64].astype(np.float32)), dtype=torch.float32).to(DEV)
                p.append(torch.softmax(model(x).float(), 1)[:, 1].cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

BANDS = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
def band_powers(W):
    fr, P = ss.welch(W, fs=SFREQ, nperseg=400, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)
    return np.concatenate([np.log(bp + 1e-12).mean(0).ravel(), (bp / bp.sum(-1, keepdims=True)).mean(0).ravel()])

print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

## 2. NMT: всё train + eval, параллельная подкачка файлов

In [ ]:
from concurrent.futures import ThreadPoolExecutor

ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
print(pd.crosstab(lab["loc"], lab["label"]))

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, lab.path)):
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT подкачано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)

ELECTRODES = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "F7", "F8", "T3", "T4", "T5", "T6", "O1", "O2", "Fz", "Cz", "Pz"]
ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in ELECTRODES]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

def load_nmt(path, max_win=None, rng=None):
    raw = mne.io.read_raw_edf(path, preload=True)
    if raw.info["sfreq"] != SFREQ:
        raw.resample(SFREQ)
    names = {norm(c): c for c in raw.ch_names}
    X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6
    n = X.shape[1] // L
    W = X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)
    if max_win and n > max_win:
        W = W[np.sort(rng.choice(n, max_win, replace=False))]
    return W

rng = np.random.default_rng(0)
NMT_W, NMT_BP = {}, {}
t0 = time.time()
for i, r in lab.iterrows():
    try:
        W = load_nmt(r.path, max_win=None if r["loc"] == "eval" else 8, rng=rng)
    except Exception:
        continue
    if len(W) == 0:
        continue
    NMT_W[r.recordname] = W.astype(np.float16); NMT_BP[r.recordname] = band_powers(W)
    if (i + 1) % 400 == 0 or i + 1 == len(lab):
        print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT_W)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))

label  abnormal  normal
loc                    
eval         90      95
train       325    1907
NMT подкачано 2417/2417 55 с
NMT прочитано 2417/2417 206 с
label  abnormal  normal
loc                    
eval         90      94
train       323    1902


## 3. Турецкий набор: 19 каналов в порядке авторов, средний референт, 200 Гц, окна 30 с

In [ ]:
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 40 == 0 or i + 1 == len(tfiles):
            print(f"Турция подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
# индексы турецких каналов в порядке авторов:     Fp1 Fp2 F3 F4 C3 C4 P3 P4 F7  F8  T3  T4  T5  T6  O1  O2  Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 13, 14, 15, 16, 17, 18, 11, 12, 4, 7, 10]

TR_W, TR_BP, y_tr = [], [], []
t0 = time.time()
for i, (f, yy) in enumerate(tfiles):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)                                                       # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)                                                  # 200 Гц, (куски, 19, 3000)
    n = len(X) // 2
    W = np.concatenate([X[0:2*n:2], X[1:2*n:2]], axis=-1)                                  # (окна, 19, 6000)
    TR_W.append(W.astype(np.float16)); TR_BP.append(band_powers(W)); y_tr.append(yy)
    if (i + 1) % 20 == 0 or i + 1 == len(tfiles):
        print(f"Турция {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
y_tr = np.array(y_tr)
print("эпилепсия:", int(y_tr.sum()), "| здоровые:", int((1 - y_tr).sum()))

Турция подкачано 121/121 27 с
Турция 121/121 498 с
эпилепсия: 50 | здоровые: 71


## 4. Дообучение: 2 варианта (без аугментаций / с аугментациями) × 3 запуска

Для каждого запуска эпоха выбирается по 15 % записей NMT train (одно и то же деление для всех запусков); турецкий набор и NMT eval при выборе не используются.

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
Xw = np.concatenate([NMT_W[n] for n in fit_names])
yw = np.concatenate([np.full(len(NMT_W[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор эпохи: {len(val_names)}")

MIRROR = [ELECTRODES.index(c) for c in ["Fp2", "Fp1", "F4", "F3", "C4", "C3", "P4", "P3", "F8", "F7", "T4", "T3", "T6", "T5", "O2", "O1", "Fz", "Cz", "Pz"]]

def augment(x, g):
    """x: (B, 19, L) после нормировки, torch на GPU."""
    B = x.shape[0]
    x = x + 0.1 * torch.randn_like(x)                                           # шум
    m = torch.rand(B, device=x.device) < 0.5                                     # зеркальная замена полушарий
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if g.random() < 0.5:                                                     # зануление 1–2 каналов
            x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)  # сдвиг до 2 с
    return x

def train_run(seed, aug, epochs=6, bs=32):
    torch.manual_seed(seed); g = np.random.default_rng(seed)
    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    w = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum())
    best_auc, best_state, best_ep, hist = -1, None, 0, []
    for ep in range(epochs):
        model.train()
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w / w.sum())
        for b in range(0, len(idx), bs):
            j = idx[b:b + bs]
            x = torch.tensor(scale(Xw[j].astype(np.float32)), dtype=torch.float32).to(DEV)
            if aug:
                x = augment(x, g)
            t = torch.tensor(yw[j], dtype=torch.long).to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                loss = F.cross_entropy(model(x), t)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        auc = roc_auc_score(y_val, predict(model, [NMT_W[n] for n in val_names]))
        hist.append(round(auc, 3))
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_ep, hist

RES, SCORES = [], {}
t0 = time.time()
for aug in [False, True]:
    for seed in [0, 1, 2]:
        model, best_ep, hist = train_run(seed, aug)
        s_ev = predict(model, [NMT_W[n] for n in ev.recordname]); s_tr = predict(model, TR_W)
        SCORES[(aug, seed)] = (s_ev, s_tr)
        if aug and seed == 0:
            torch.save(model.state_dict(), "/kaggle/working/deepepilepsy_nmt_aug.pt")
        RES.append(dict(aug=aug, seed=seed, best_epoch=best_ep, val_by_epoch=hist,
                        nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr)))
        print(f"{'аугм.' if aug else 'без аугм.':9s} запуск {seed}: эпоха {best_ep}, отложенные NMT по эпохам {hist} | "
              f"NMT eval {RES[-1]['nmt_eval']:.3f} | Турция {RES[-1]['turkey']:.3f} | {time.time()-t0:.0f} с", flush=True)
res = pd.DataFrame(RES)
res.to_csv("/kaggle/working/deepepilepsy_v2_runs.csv", index=False)

окон для обучения: 15007 (патология 0.14) | записей на выбор эпохи: 334
без аугм. запуск 0: эпоха 3, отложенные NMT по эпохам [np.float64(0.864), np.float64(0.864), np.float64(0.866), np.float64(0.856), np.float64(0.777), np.float64(0.773)] | NMT eval 0.862 | Турция 0.751 | 233 с
без аугм. запуск 1: эпоха 3, отложенные NMT по эпохам [np.float64(0.835), np.float64(0.862), np.float64(0.865), np.float64(0.821), np.float64(0.841), np.float64(0.773)] | NMT eval 0.822 | Турция 0.636 | 465 с
без аугм. запуск 2: эпоха 1, отложенные NMT по эпохам [np.float64(0.861), np.float64(0.827), np.float64(0.829), np.float64(0.855), np.float64(0.841), np.float64(0.56)] | NMT eval 0.803 | Турция 0.766 | 694 с
аугм.     запуск 0: эпоха 3, отложенные NMT по эпохам [np.float64(0.838), np.float64(0.86), np.float64(0.874), np.float64(0.872), np.float64(0.874), np.float64(0.872)] | NMT eval 0.887 | Турция 0.709 | 936 с
аугм.     запуск 1: эпоха 6, отложенные NMT по эпохам [np.float64(0.827), np.float64(0.863), n

## 5. Итоги: среднее ± разброс по запускам, ансамбль запусков, ансамбль с простыми признаками

In [ ]:
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

print(res.groupby("aug")[["nmt_eval", "turkey"]].agg(["mean", "std"]).round(3))

def rank(s): return rankdata(s) / len(s)

# простая модель: относительные мощности ритмов (без амплитуды), обучение на NMT train
Xb_nmt = np.stack([NMT_BP[n] for n in tr.recordname]); half = Xb_nmt.shape[1] // 2
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000, class_weight="balanced"))
clf.fit(Xb_nmt[:, half:], tr.y)
bp_tr = clf.predict_proba(np.stack(TR_BP)[:, half:])[:, 1]
bp_ev = clf.predict_proba(np.stack([NMT_BP[n] for n in ev.recordname])[:, half:])[:, 1]

rows = []
for aug in [False, True]:
    ens_ev = np.mean([rank(SCORES[(aug, s)][0]) for s in range(3)], 0)
    ens_tr = np.mean([rank(SCORES[(aug, s)][1]) for s in range(3)], 0)
    name = "с аугм." if aug else "без аугм."
    for title, se, st in [(f"DeepEpilepsy {name}, ансамбль 3 запусков", ens_ev, ens_tr),
                          (f"DeepEpilepsy {name} + относит. мощности", (ens_ev + rank(bp_ev)) / 2, (ens_tr + rank(bp_tr)) / 2)]:
        lo, hi = boot_ci(y_tr, st)
        rows.append(dict(модель=title, NMT_eval=round(roc_auc_score(ev.y, se), 3),
                         Турция=round(roc_auc_score(y_tr, st), 3), Турция_95ДИ=f"{lo:.2f}–{hi:.2f}"))
lo, hi = boot_ci(y_tr, bp_tr)
rows.append(dict(модель="Относительные мощности ритмов", NMT_eval=round(roc_auc_score(ev.y, bp_ev), 3),
                 Турция=round(roc_auc_score(y_tr, bp_tr), 3), Турция_95ДИ=f"{lo:.2f}–{hi:.2f}"))
summary = pd.DataFrame(rows)
summary.to_csv("/kaggle/working/deepepilepsy_v2_summary.csv", index=False)
print(summary.to_string(index=False))

      nmt_eval        turkey       
          mean    std   mean    std
aug                                
False    0.829  0.030  0.718  0.071
True     0.891  0.008  0.719  0.012
                                     модель  NMT_eval  Турция Турция_95ДИ
DeepEpilepsy без аугм., ансамбль 3 запусков     0.842   0.727   0.64–0.81
 DeepEpilepsy без аугм. + относит. мощности     0.863   0.751   0.66–0.83
  DeepEpilepsy с аугм., ансамбль 3 запусков     0.900   0.729   0.64–0.81
   DeepEpilepsy с аугм. + относит. мощности     0.882   0.719   0.63–0.81
              Относительные мощности ритмов     0.829   0.655   0.55–0.76


## 6. Выводы

**Запуски по отдельности** (эпоха выбрана по отложенным записям NMT train):

| Вариант | NMT eval, среднее ± разброс | Турция, среднее ± разброс | Лучшая эпоха |
|---|---|---|---|
| Без аугментаций | 0,83 ± 0,03 (0,80–0,86) | 0,72 ± 0,07 (0,64–0,77) | 1–3, дальше переобучение |
| **С аугментациями** | **0,89 ± 0,01** (0,89–0,90) | **0,72 ± 0,01** (0,71–0,73) | 3–6, качество растёт до конца |

**Ансамбли** (среднее рангов предсказаний 3 запусков):

| Модель | NMT eval | Турция (95% ДИ) |
|---|---|---|
| DeepEpilepsy без аугм., ансамбль | 0,84 | 0,73 (0,64–0,81) |
| **DeepEpilepsy с аугм., ансамбль** | **0,90** | **0,73 (0,64–0,81)** |
| DeepEpilepsy без аугм. + относит. мощности | 0,86 | 0,75 (0,66–0,83) |
| DeepEpilepsy с аугм. + относит. мощности | 0,88 | 0,72 (0,63–0,81) |
| Относительные мощности ритмов | 0,83 | 0,66 (0,55–0,76) |

1. **Всё NMT вместо подвыборки подняло перенос в Турцию с 0,66 до 0,72–0,73.**
2. **Аугментации сильно помогают**: NMT eval 0,83 → 0,89–0,90, а разброс между запусками упал в 3–7 раз (на NMT 0,03 → 0,01, на Турции 0,07 → 0,01). Без аугментаций модель переобучается после 1–3 эпох; с аугментациями качество растёт до 6-й эпохи — есть запас на более длинное обучение.
3. **DeepEpilepsy теперь явно лучше простой модели**: 0,90 против 0,83 на NMT и 0,73 против 0,66 на Турции.
4. Ансамбль с мощностями ритмов даёт небольшую прибавку только без аугментаций; с аугментациями — не нужен.
5. **Рабочая конфигурация для TUEP**: всё смежное клиническое (NMT, затем TUAB) + аугментации, ансамбль 3 запусков; веса после NMT сохранены (`deepepilepsy_nmt_aug.pt`, в git не кладём).

**Ограничения:** задачи разные (NMT — «патология», Турция — «эпилепсия»); контроль в турецком наборе — здоровые; 3 запуска; интервал по Турции широкий (121 человек). Время на Kaggle — около 40 мин GPU.

## 7. Что дальше

- Больше эпох (8–12) с аугментациями; затем TUAB → TUEP, когда будет доступ.
- То же сравнение для CBraMod и REVE (в работе) и детектор разрядов как второй признак (в работе).
